### Silver local

Notebook para construir a camada **Silver** a partir da camada Bronze.

Principais correções:

- resultados em formato longo usando `ano_da_avaliacao`;
- prioridade explícita para `percentual_de_alunos_alfabetizados_1`;
- gravação incremental real por **upsert** e deduplicação por chave;
- chamada de execução fora de `main()`, evitando recursão infinita;
- validação de arquivos e colunas obrigatórias;
- tratamento mais seguro de códigos de município, arquivos ausentes e valores numéricos;
- relatório final com arquivos gerados e eventuais erros.

> Ajuste `BASE_DIR` antes de executar. As células foram organizadas para execução sequencial.


In [28]:
from __future__ import annotations

import gc
import json
import re
import traceback
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Iterable

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


In [29]:
# Caminhos e parâmetros
BASE_DIR = Path("/home/linux/projetos/Teste_02112026").resolve()
BRONZE_DIR = BASE_DIR / "data" / "bronze"
SILVER_DIR = BASE_DIR / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

ANOS = (2023, 2024, 2025)
AGORA = datetime.now(timezone.utc).isoformat()

if not BRONZE_DIR.exists():
    raise FileNotFoundError(f"Diretório Bronze não encontrado: {BRONZE_DIR}")

print(f"BASE_DIR:   {BASE_DIR}")
print(f"BRONZE_DIR: {BRONZE_DIR}")
print(f"SILVER_DIR: {SILVER_DIR}")


BASE_DIR:   /home/linux/projetos/Teste_02112026
BRONZE_DIR: /home/linux/projetos/Teste_02112026/data/bronze
SILVER_DIR: /home/linux/projetos/Teste_02112026/data/silver


In [30]:
# Utilitários gerais
def norm(valor) -> str:
    texto = "".join(
        caractere
        for caractere in unicodedata.normalize("NFKD", str(valor))
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^a-z0-9]+", "_", texto.strip().lower()).strip("_")


def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    usados: dict[str, int] = {}
    colunas: list[str] = []

    for coluna in df.columns:
        base = norm(coluna) or "coluna"
        usados[base] = usados.get(base, 0) + 1
        colunas.append(base if usados[base] == 1 else f"{base}_{usados[base]}")

    df.columns = colunas
    return df


def txt(serie: pd.Series) -> pd.Series:
    return (
        serie.astype("string")
        .str.strip()
        .replace({"": pd.NA, "-": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def num(serie: pd.Series) -> pd.Series:
    texto = txt(serie)
    usa_virgula = texto.str.contains(",", na=False)
    texto.loc[usa_virgula] = (
        texto.loc[usa_virgula]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(texto, errors="coerce")


def cod_mun(serie: pd.Series) -> pd.Series:
    codigo = txt(serie)
    codigo = codigo.str.replace(r"\.0$", "", regex=True)
    codigo = codigo.str.replace(r"\D", "", regex=True)
    codigo = codigo.where(codigo.ne(""), pd.NA)
    return codigo.str.zfill(7)


def aliases(df: pd.DataFrame, mapa: dict[str, list[str]]) -> pd.DataFrame:
    renomear: dict[str, str] = {}
    for destino, candidatos in mapa.items():
        if destino in df.columns:
            continue
        for candidato in candidatos:
            if candidato in df.columns:
                renomear[candidato] = destino
                break
    return df.rename(columns=renomear)


def validar_colunas(df: pd.DataFrame, obrigatorias: Iterable[str], contexto: str) -> None:
    ausentes = [coluna for coluna in obrigatorias if coluna not in df.columns]
    if ausentes:
        raise KeyError(f"{contexto}: colunas obrigatórias ausentes: {ausentes}")


In [31]:
# Chaves usadas no upsert incremental de cada arquivo Silver
CHAVES_ARQUIVO: dict[str, list[str]] = {
    "dim_estados.parquet": ["id_uf"],
    "dim_municipios.parquet": ["id_municipio"],
    "censo_2022_municipios.parquet": ["id_municipio", "ano_referencia"],
    "pib_municipal.parquet": ["id_municipio", "ano_pib"],
    "inse_2023_municipios.parquet": ["id_municipio", "rede", "ano_referencia"],
    "ideb_anos_iniciais.parquet": ["id_municipio", "rede", "ano"],
    "municipio.parquet": ["id_municipio", "rede", "ano"],
    "uf.parquet": ["sigla_uf", "rede", "ano"],
    "alunos.parquet": ["id_aluno", "ano"],
    "resultados_municipios.parquet": ["id_municipio", "rede", "ano"],
    "metas_municipio.parquet": ["id_municipio", "rede", "ano"],
    "resultados_uf.parquet": ["sigla_uf", "rede", "ano"],
    "metas_uf.parquet": ["sigla_uf", "rede", "ano"],
    "meta_brasil.parquet": ["sigla_uf", "rede", "ano"],
}


def gravar_incremental(
    df_novo: pd.DataFrame,
    nome: str,
    chaves: list[str] | None = None,
) -> Path:
    if df_novo is None or df_novo.empty:
        raise ValueError(f"{nome}: DataFrame vazio")

    caminho = SILVER_DIR / nome
    chaves = chaves or CHAVES_ARQUIVO.get(nome)

    novo = df_novo.copy()
    novo["_processed_at"] = AGORA

    if caminho.exists():
        existente = pd.read_parquet(caminho, engine="pyarrow")
        colunas = existente.columns.union(novo.columns, sort=False)
        existente = existente.reindex(columns=colunas)
        novo = novo.reindex(columns=colunas)
        consolidado = pd.concat([existente, novo], ignore_index=True, sort=False)
        modo = "upsert"
    else:
        consolidado = novo
        modo = "criação"

    if chaves:
        chaves_validas = [chave for chave in chaves if chave in consolidado.columns]
        if not chaves_validas:
            raise KeyError(f"{nome}: nenhuma chave de upsert foi encontrada: {chaves}")

        chaves_ausentes = [chave for chave in chaves if chave not in consolidado.columns]
        if chaves_ausentes:
            print(f"[AVISO] {nome}: chaves ausentes ignoradas: {chaves_ausentes}")

        consolidado = (
            consolidado.sort_values("_processed_at", kind="stable")
            .drop_duplicates(subset=chaves_validas, keep="last")
        )
    else:
        print(f"[AVISO] {nome}: sem chave configurada; removendo apenas linhas idênticas")
        colunas_dados = [c for c in consolidado.columns if c != "_processed_at"]
        consolidado = consolidado.drop_duplicates(subset=colunas_dados, keep="last")

    temporario = caminho.with_suffix(".tmp.parquet")
    consolidado.to_parquet(
        temporario,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )
    temporario.replace(caminho)

    tamanho_mb = caminho.stat().st_size / 1024**2
    print(f"[OK] {nome}: {len(consolidado):,} linhas | {tamanho_mb:.2f} MB | {modo}")
    return caminho


def ler_csv(caminho: Path) -> pd.DataFrame:
    ultimo_erro: Exception | None = None
    for encoding in ("latin-1", "utf-8", "utf-8-sig", "cp1252"):
        try:
            return pd.read_csv(caminho, sep=";", encoding=encoding, low_memory=False)
        except UnicodeDecodeError as erro:
            ultimo_erro = erro
    raise RuntimeError(f"Falha ao ler {caminho}") from ultimo_erro


def localizar_csv(pasta: Path, termos: Iterable[str]) -> Path | None:
    if not pasta.exists():
        return None
    candidatos = [
        caminho
        for caminho in pasta.rglob("*.csv")
        if any(termo in norm(caminho.stem) for termo in termos)
    ]
    return sorted(candidatos, key=lambda p: (len(p.parts), p.name))[0] if candidatos else None


def detectar_header(caminho: Path, tokens: Iterable[str]) -> int:
    amostra = pd.read_excel(caminho, header=None, nrows=150, engine="openpyxl")
    tokens_normalizados = [norm(token) for token in tokens]

    for indice, linha in amostra.iterrows():
        valores = {norm(valor) for valor in linha.dropna()}
        if all(any(token == valor or token in valor for valor in valores) for token in tokens_normalizados):
            return int(indice)

    raise ValueError(f"Cabeçalho não localizado em {caminho.name}")


def ler_excel(caminho: Path, tokens: Iterable[str]) -> pd.DataFrame:
    if not caminho.exists():
        raise FileNotFoundError(caminho)
    header = detectar_header(caminho, tokens)
    df = pd.read_excel(caminho, header=header, engine="openpyxl", dtype="object")
    return normalizar_colunas(df).dropna(how="all")


In [32]:
# Microdados
A_ALUNO = {
    "sigla_uf": ["sg_uf", "uf"],
    "id_municipio": ["co_municipio", "codigo_municipio"],
    "id_escola": ["co_escola"],
    "id_aluno": ["co_aluno", "id_estudante"],
    "rede": ["tp_rede", "rede_ensino"],
    "presente": ["in_presenca", "presenca", "in_presenca_lp"],
    "preencheu": ["in_preenchimento_caderno", "preenchimento_caderno"],
    "alfabetizado": ["in_alfabetizado"],
    "proficiencia": ["vl_proficiencia", "vl_proficiencia_lp"],
    "peso": ["peso_aluno", "vl_peso", "peso_lp"],
}

A_MUN = {
    "sigla_uf": ["sg_uf", "uf"],
    "id_municipio": ["co_municipio"],
    "nome_municipio": ["no_municipio"],
    "rede": ["tp_rede"],
    "indicador_pct": ["pc_aluno_alfabetizado", "pc_alfabetizados"],
    "participacao_pct": ["pc_participacao"],
    "nivel": ["nu_nivel"],
}

A_UF = {
    "sigla_uf": ["sg_uf", "uf"],
    "id_uf": ["co_uf"],
    "nome_uf": ["no_uf", "estado"],
    "rede": ["tp_rede"],
    "indicador_pct": ["pc_aluno_alfabetizado", "pc_alfabetizados"],
    "participacao_pct": ["pc_participacao"],
    "nivel": ["nu_nivel"],
}


def micro(tipo: str, mapa: dict[str, list[str]]) -> pd.DataFrame:
    termos_por_tipo = {
        "aluno": ("ts_aluno", "aluno"),
        "municipio": ("ts_municipio", "municipio"),
        "uf": ("ts_estado", "ts_uf", "estado"),
    }
    if tipo not in termos_por_tipo:
        raise ValueError(f"Tipo de microdado inválido: {tipo}")

    partes: list[pd.DataFrame] = []
    for ano in ANOS:
        caminho = localizar_csv(BRONZE_DIR / f"microdados_{ano}", termos_por_tipo[tipo])
        if caminho is None:
            print(f"[AVISO] {tipo} {ano} ausente")
            continue

        print(f"Lendo {tipo} {ano}: {caminho.relative_to(BASE_DIR)}")
        dados = aliases(normalizar_colunas(ler_csv(caminho)), mapa)
        dados["ano"] = ano
        dados["_arquivo_fonte"] = caminho.name
        partes.append(dados)

    if not partes:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para microdados de {tipo}")

    dados = pd.concat(partes, ignore_index=True, sort=False)

    if "id_municipio" in dados.columns:
        dados["id_municipio"] = cod_mun(dados["id_municipio"])

    for coluna in ("sigla_uf", "nome_uf", "nome_municipio", "id_aluno", "id_escola", "rede"):
        if coluna in dados.columns:
            dados[coluna] = txt(dados[coluna])

    for coluna in (
        "presente", "preencheu", "alfabetizado", "indicador_pct",
        "participacao_pct", "nivel", "proficiencia", "peso",
    ):
        if coluna in dados.columns:
            dados[coluna] = num(dados[coluna])

    return dados


In [33]:
# Resultados e metas em Excel
def ano_col(coluna: str) -> int | None:
    anos = re.findall(r"(20\d{2})", norm(coluna))
    return int(anos[-1]) if anos else None


def e_meta(coluna: str) -> bool:
    return any(termo in norm(coluna) for termo in ("meta", "projecao", "projetado", "trajetoria"))


def e_resultado(coluna: str) -> bool:
    nome = norm(coluna)
    ano = ano_col(coluna)
    if ano is None or e_meta(coluna) or any(x in nome for x in ("nivel", "participacao", "saeb")):
        return False
    return nome == str(ano) or any(x in nome for x in ("resultado", "indicador", "percentual", "alfabetizado"))


def preparar_resultado(caminho: Path, ano_fonte: int, nivel: str) -> pd.DataFrame:
    if nivel not in {"municipio", "uf"}:
        raise ValueError(f"Nível inválido: {nivel}")

    try:
        tokens = ("municipio", "uf") if nivel == "municipio" else ("uf",)
        dados = ler_excel(caminho, tokens)
    except ValueError:
        tokens = ("codigo",) if nivel == "municipio" else ("sigla",)
        dados = ler_excel(caminho, tokens)

    dados = aliases(
        dados,
        {
            "id_municipio": ["co_municipio", "codigo_do_municipio", "codigo_municipio"],
            "nome_municipio": ["no_municipio", "nome_do_municipio", "municipio"],
            "sigla_uf": ["sg_uf", "sigla_da_uf", "uf"],
            "nome_uf": ["no_uf", "nome_da_uf", "estado"],
            "rede": ["rede_de_ensino", "dependencia_administrativa"],
        },
    )

    if "id_municipio" in dados.columns:
        dados["id_municipio"] = cod_mun(dados["id_municipio"])

    dados["ano_fonte"] = ano_fonte
    dados["_arquivo_fonte"] = caminho.name
    return dados


def localizar_colunas_resultado(dados: pd.DataFrame) -> list[str]:
    # Prioridade solicitada: a segunda coluna duplicada normalizada recebe sufixo _1.
    preferidas = [
        "percentual_de_alunos_alfabetizados_1",
        "percentual_de_alunos_alfabetizados",
        "percentual_alfabetizados_1",
        "percentual_alfabetizados",
    ]
    encontradas = [coluna for coluna in preferidas if coluna in dados.columns and not e_meta(coluna)]
    if encontradas:
        return [encontradas[0]]

    por_ano = [coluna for coluna in dados.columns if e_resultado(coluna)]
    if por_ano:
        return por_ano

    fallback = [
        coluna
        for coluna in dados.columns
        if any(
            termo in norm(coluna)
            for termo in (
                "percentual_de_alunos_alfabetizados",
                "percentual_alfabetizados",
                "alunos_alfabetizados",
            )
        )
        and not e_meta(coluna)
    ]
    return fallback[:1]


def longos(dados: pd.DataFrame, nivel: str, metas: bool = False) -> pd.DataFrame:
    chaves_descritivas = [
        coluna
        for coluna in ("id_municipio", "nome_municipio", "sigla_uf", "nome_uf", "rede")
        if coluna in dados.columns
    ]
    ano_fonte = int(dados["ano_fonte"].iloc[0])
    arquivo_fonte = dados["_arquivo_fonte"].iloc[0]

    if metas:
        colunas_meta = [coluna for coluna in dados.columns if e_meta(coluna) and ano_col(coluna)]
        if not colunas_meta:
            raise ValueError(f"Nenhuma coluna de meta em {arquivo_fonte}. Colunas: {list(dados.columns)}")

        partes: list[pd.DataFrame] = []
        for coluna in colunas_meta:
            parte = dados[chaves_descritivas].copy()
            parte["ano"] = ano_col(coluna)
            parte["meta_pct"] = num(dados[coluna])
            parte["ano_fonte"] = ano_fonte
            parte["_arquivo_fonte"] = dados["_arquivo_fonte"].values
            parte["_coluna_origem"] = coluna
            partes.append(parte)

        saida = pd.concat(partes, ignore_index=True, sort=False)
        saida.loc[saida["meta_pct"].eq(0), "meta_pct"] = pd.NA
        saida = saida.dropna(subset=["ano", "meta_pct"])
        saida["ano"] = pd.to_numeric(saida["ano"], errors="coerce").astype("Int16")
        return saida

    coluna_ano = next(
        (coluna for coluna in ("ano_da_avaliacao", "ano_avaliacao", "ano") if coluna in dados.columns),
        None,
    )
    colunas_resultado = localizar_colunas_resultado(dados)

    if not colunas_resultado:
        raise ValueError(f"Nenhuma coluna de resultado em {arquivo_fonte}. Colunas: {list(dados.columns)}")

    nivel_col = next((c for c in dados.columns if "nivel_alfabetizacao" in norm(c)), None)
    participacao_col = next(
        (
            c for c in dados.columns
            if "percentual_de_participacao" in norm(c) or norm(c) == "participacao_pct"
        ),
        None,
    )

    partes = []
    for coluna in colunas_resultado:
        parte = dados[chaves_descritivas].copy()
        ano_no_nome = ano_col(coluna)

        if ano_no_nome is not None:
            parte["ano"] = ano_no_nome
        elif coluna_ano is not None:
            parte["ano"] = num(dados[coluna_ano]).astype("Int64")
        else:
            parte["ano"] = ano_fonte

        parte["indicador_pct"] = num(dados[coluna])

        if nivel_col:
            parte["nivel"] = num(dados[nivel_col])
        if participacao_col:
            parte["participacao_pct"] = num(dados[participacao_col])

        parte["ano_fonte"] = ano_fonte
        parte["_arquivo_fonte"] = dados["_arquivo_fonte"].values
        parte["_coluna_origem"] = coluna
        partes.append(parte)

    saida = pd.concat(partes, ignore_index=True, sort=False)
    saida = saida.dropna(subset=["ano", "indicador_pct"])
    saida["ano"] = pd.to_numeric(saida["ano"], errors="coerce").astype("Int16")
    return saida


def consolidar_planilhas(nivel: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    resultados: list[pd.DataFrame] = []
    metas: list[pd.DataFrame] = []
    plural = "municipios" if nivel == "municipio" else "ufs"

    for ano in ANOS:
        caminho = BRONZE_DIR / f"resultados_{plural}_{ano}.xlsx"
        if not caminho.exists():
            print(f"[AVISO] Arquivo ausente: {caminho.name}")
            continue

        dados = preparar_resultado(caminho, ano, nivel)
        resultados.append(longos(dados, nivel))
        metas.append(longos(dados, nivel, metas=True))

    if not resultados:
        raise FileNotFoundError(f"Nenhuma planilha de resultados encontrada para {nivel}")
    if not metas:
        raise ValueError(f"Nenhuma meta encontrada para {nivel}")

    resultado = pd.concat(resultados, ignore_index=True, sort=False)
    meta = pd.concat(metas, ignore_index=True, sort=False)

    chave_base = ["id_municipio"] if nivel == "municipio" else ["sigla_uf"]
    chave_resultado = chave_base + ["ano"]
    chave_meta = chave_base + ["ano"]

    if "rede" in resultado.columns:
        chave_resultado.append("rede")
    if "rede" in meta.columns:
        chave_meta.append("rede")

    resultado = (
        resultado.sort_values("ano_fonte", kind="stable")
        .drop_duplicates(chave_resultado, keep="last")
    )
    meta = (
        meta.sort_values("ano_fonte", kind="stable")
        .drop_duplicates(chave_meta, keep="last")
    )
    return resultado, meta


In [34]:
# IBGE, SIDRA, INSE e IDEB
def loadj(nome: str):
    caminho = BRONZE_DIR / nome
    if not caminho.exists():
        raise FileNotFoundError(caminho)
    with caminho.open(encoding="utf-8-sig") as arquivo:
        return json.load(arquivo)


def dim_estados() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "id_uf": str(item["id"]).zfill(2),
                "sigla_uf": item["sigla"],
                "nome_uf": item["nome"],
                "nome_regiao": (item.get("regiao") or {}).get("nome"),
            }
            for item in loadj("ibge_estados.json")
        ]
    )


def dim_municipios() -> pd.DataFrame:
    linhas = []
    for item in loadj("ibge_municipios.json"):
        uf = (((item.get("microrregiao") or {}).get("mesorregiao") or {}).get("UF") or {})
        if not uf:
            uf = ((((item.get("regiao-imediata") or {}).get("regiao-intermediaria") or {}).get("UF")) or {})

        id_uf = uf.get("id")
        linhas.append(
            {
                "id_municipio": str(item["id"]).zfill(7),
                "nome_municipio": item["nome"],
                "id_uf": str(id_uf).zfill(2) if id_uf is not None else pd.NA,
                "sigla_uf": uf.get("sigla"),
                "nome_uf": uf.get("nome"),
                "nome_regiao": (uf.get("regiao") or {}).get("nome"),
            }
        )
    return pd.DataFrame(linhas)


def sidra(nome: str, mapa: dict[str, str], ano_coluna: str) -> pd.DataFrame:
    dados_json = loadj(nome)
    if len(dados_json) < 2:
        raise ValueError(f"{nome}: resposta SIDRA vazia ou inválida")

    cabecalho = dados_json[0]
    dados = normalizar_colunas(
        pd.DataFrame(
            [{cabecalho.get(chave, chave): valor for chave, valor in item.items()} for item in dados_json[1:]]
        )
    )

    coluna_municipio = next(
        c for c in dados.columns if "municipio" in c and ("codigo" in c or c.endswith("_cod"))
    )
    coluna_variavel = next(
        c for c in dados.columns if "variavel" in c and ("codigo" in c or c.endswith("_cod"))
    )
    coluna_valor = "valor"
    coluna_nome = next((c for c in dados.columns if c == "municipio"), None)

    base = pd.DataFrame(
        {
            "id_municipio": cod_mun(dados[coluna_municipio]),
            "nome_municipio": txt(dados[coluna_nome]) if coluna_nome else pd.NA,
            "codigo": txt(dados[coluna_variavel]),
            "valor": num(dados[coluna_valor]),
        }
    )
    base["campo"] = base["codigo"].map(mapa)

    saida = (
        base.dropna(subset=["campo"])
        .pivot_table(
            index=["id_municipio", "nome_municipio"],
            columns="campo",
            values="valor",
            aggfunc="first",
        )
        .reset_index()
    )
    saida.columns.name = None
    saida[ano_coluna] = 2022 if "censo" in nome else 2021
    return saida


def inse() -> pd.DataFrame:
    dados = ler_excel(BRONZE_DIR / "inse_2023_municipios.xlsx", ("municipio", "rede"))
    dados = aliases(
        dados,
        {
            "id_municipio": ["co_municipio", "codigo_do_municipio", "codigo_municipio"],
            "nome_municipio": ["no_municipio", "nome_do_municipio", "municipio"],
            "sigla_uf": ["sg_uf", "uf"],
            "rede": ["rede_de_ensino", "dependencia_administrativa"],
            "localizacao": ["localizacao_da_escola", "localizacao"],
            "inse_qtd_alunos": ["quantidade_de_alunos", "qtd_alunos", "nu_alunos"],
            "inse_medio": ["media_do_inse", "inse_medio", "valor_inse"],
        },
    )
    validar_colunas(dados, ["id_municipio"], "INSE")
    dados["id_municipio"] = cod_mun(dados["id_municipio"])
    dados["ano_referencia"] = 2023
    return dados


def ideb() -> pd.DataFrame:
    pasta = BRONZE_DIR / "ideb_anos_iniciais"
    candidatos = sorted(p for p in pasta.rglob("*.xlsx") if "municip" in norm(p.name))
    if not candidatos:
        raise FileNotFoundError(f"Nenhuma planilha municipal do IDEB em {pasta}")

    caminho = candidatos[0]
    dados = ler_excel(caminho, ("sg_uf", "co_municipio", "rede"))
    dados = aliases(
        dados,
        {
            "sigla_uf": ["sg_uf"],
            "id_municipio": ["co_municipio"],
            "nome_municipio": ["no_municipio"],
        },
    )
    validar_colunas(dados, ["sigla_uf", "id_municipio", "nome_municipio", "rede"], "IDEB")
    dados["id_municipio"] = cod_mun(dados["id_municipio"])

    mapa = {
        "vl_aprovacao_{a}_si_4": "aprovacao_ai_pct",
        "vl_aprovacao_{a}_si": "aprovacao_1ano_pct",
        "vl_aprovacao_{a}_1": "aprovacao_2ano_pct",
        "vl_indicador_rend_{a}": "ideb_rendimento",
        "vl_nota_matematica_{a}": "ideb_nota_matematica",
        "vl_nota_portugues_{a}": "ideb_nota_portugues",
        "vl_nota_media_{a}": "ideb_nota_media",
        "vl_observado_{a}": "ideb_observado",
        "vl_projecao_{a}": "ideb_meta",
    }

    partes = []
    for ano in (2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021, 2023, 2025):
        presentes = {
            origem.format(a=ano): destino
            for origem, destino in mapa.items()
            if origem.format(a=ano) in dados.columns
        }
        if not presentes:
            continue

        parte = dados[
            ["sigla_uf", "id_municipio", "nome_municipio", "rede", *presentes.keys()]
        ].rename(columns=presentes).copy()
        parte["ano"] = ano
        for coluna in presentes.values():
            parte[coluna] = num(parte[coluna])
        partes.append(parte)

    if not partes:
        raise ValueError(f"Nenhuma coluna anual do IDEB reconhecida em {caminho.name}")
    return pd.concat(partes, ignore_index=True, sort=False)


In [35]:
# Orquestração incremental
def main() -> list[tuple[str, str]]:
    erros: list[tuple[str, str]] = []

    def etapa(nome: str, arquivo: str, funcao: Callable[[], pd.DataFrame]) -> None:
        print(f"\n--- {nome} ---")
        try:
            dados = funcao()
            gravar_incremental(dados, arquivo)
            del dados
            gc.collect()
        except Exception as erro:
            erros.append((nome, str(erro)))
            traceback.print_exc()

    etapa("Dimensão estados", "dim_estados.parquet", dim_estados)
    etapa("Dimensão municípios", "dim_municipios.parquet", dim_municipios)
    etapa(
        "Censo 2022",
        "censo_2022_municipios.parquet",
        lambda: sidra(
            "censo_2022.json",
            {"93": "populacao", "6318": "area_km2", "614": "densidade_hab_km2"},
            "ano_referencia",
        ),
    )
    etapa(
        "PIB municipal",
        "pib_municipal.parquet",
        lambda: sidra(
            "pib_municipal.json",
            {
                "37": "pib_mil_reais",
                "516": "vab_agro_mil_reais",
                "520": "vab_industria_mil_reais",
                "528": "vab_servicos_mil_reais",
                "6574": "vab_adm_publica_mil_reais",
            },
            "ano_pib",
        ),
    )
    etapa("INSE", "inse_2023_municipios.parquet", inse)
    etapa("IDEB", "ideb_anos_iniciais.parquet", ideb)
    etapa("Microdados município", "municipio.parquet", lambda: micro("municipio", A_MUN))
    etapa("Microdados UF", "uf.parquet", lambda: micro("uf", A_UF))

    print("\n--- Resultados e metas municipais ---")
    try:
        resultados, metas = consolidar_planilhas("municipio")
        gravar_incremental(resultados, "resultados_municipios.parquet")
        gravar_incremental(metas, "metas_municipio.parquet")
        del resultados, metas
        gc.collect()
    except Exception as erro:
        erros.append(("Resultados/metas municipais", str(erro)))
        traceback.print_exc()

    print("\n--- Resultados e metas de UF/Brasil ---")
    try:
        resultados, metas = consolidar_planilhas("uf")

        sigla_resultado = resultados.get(
            "sigla_uf", pd.Series(pd.NA, index=resultados.index, dtype="string")
        ).astype("string").str.upper()
        nome_resultado = resultados.get(
            "nome_uf", pd.Series(pd.NA, index=resultados.index, dtype="string")
        ).astype("string").str.upper()
        brasil_resultado = sigla_resultado.eq("BR") | nome_resultado.eq("BRASIL")

        gravar_incremental(resultados.loc[~brasil_resultado].copy(), "resultados_uf.parquet")

        sigla_meta = metas.get(
            "sigla_uf", pd.Series(pd.NA, index=metas.index, dtype="string")
        ).astype("string").str.upper()
        nome_meta = metas.get(
            "nome_uf", pd.Series(pd.NA, index=metas.index, dtype="string")
        ).astype("string").str.upper()
        brasil_meta = sigla_meta.eq("BR") | nome_meta.eq("BRASIL")

        gravar_incremental(metas.loc[~brasil_meta].copy(), "metas_uf.parquet")
        gravar_incremental(metas.loc[brasil_meta].copy(), "meta_brasil.parquet")
        del resultados, metas
        gc.collect()
    except Exception as erro:
        erros.append(("Resultados/metas UF e Brasil", str(erro)))
        traceback.print_exc()

    etapa("Microdados alunos", "alunos.parquet", lambda: micro("aluno", A_ALUNO))

    arquivos = sorted(SILVER_DIR.glob("*.parquet"))
    print(f"\nTotal: {len(arquivos)}/14 arquivos Parquet")
    for arquivo in arquivos:
        print(f"- {arquivo.name}")

    if erros:
        print("\n[AVISO] A Silver foi concluída parcialmente:")
        for etapa_nome, mensagem in erros:
            print(f"- {etapa_nome}: {mensagem}")
    else:
        print("\n[OK] Camada Silver concluída sem erros.")

    return erros


## Executar a pipeline

A chamada está em uma célula separada. Isso evita a recursão infinita que ocorria quando `main()` era chamado dentro da própria função.


In [36]:
erros = main()

if erros:
    raise RuntimeError(f"Silver parcial: {erros}")



--- Dimensão estados ---
[OK] dim_estados.parquet: 27 linhas | 0.00 MB | upsert

--- Dimensão municípios ---
[OK] dim_municipios.parquet: 5,571 linhas | 0.10 MB | upsert

--- Censo 2022 ---
[OK] censo_2022_municipios.parquet: 5,570 linhas | 0.20 MB | upsert

--- PIB municipal ---
[OK] pib_municipal.parquet: 5,570 linhas | 0.23 MB | upsert

--- INSE ---
[AVISO] inse_2023_municipios.parquet: chaves ausentes ignoradas: ['rede']
[OK] inse_2023_municipios.parquet: 5,558 linhas | 0.28 MB | upsert

--- IDEB ---
[OK] ideb_anos_iniciais.parquet: 159,698 linhas | 3.68 MB | upsert

--- Microdados município ---
Lendo municipio 2023: data/bronze/microdados_2023/DADOS/TS_MUNICIPIO.csv
Lendo municipio 2024: data/bronze/microdados_2024/DADOS/TS_MUNICIPIO.csv
Lendo municipio 2025: data/bronze/microdados_2025/DADOS/TS_MUNICIPIO.csv
[AVISO] municipio.parquet: chaves ausentes ignoradas: ['rede']
[OK] municipio.parquet: 16,583 linhas | 0.53 MB | upsert

--- Microdados UF ---
Lendo uf 2023: data/bronze/mic

## Conferência dos arquivos gerados


In [37]:
resumo_arquivos = pd.DataFrame(
    [
        {
            "arquivo": caminho.name,
            "tamanho_mb": round(caminho.stat().st_size / 1024**2, 2),
            "linhas": len(pd.read_parquet(caminho, columns=None)),
        }
        for caminho in sorted(SILVER_DIR.glob("*.parquet"))
    ]
)

resumo_arquivos


,arquivo,tamanho_mb,linhas
0,alunos.parquet,85.96,6090791
1,censo_2022_municipios.parquet,0.20,5570
2,dim_estados.parquet,0.00,27
3,dim_municipios.parquet,0.10,5571
4,ideb_anos_iniciais.parquet,3.68,159698
5,inse_2023_municipios.parquet,0.28,5558
6,meta_brasil.parquet,0.01,6
7,metas_municipio.parquet,0.23,38334
8,metas_uf.parquet,0.01,160
9,municipio.parquet,0.53,16583


## Validação específica dos resultados municipais

Esta célula confirma se o resultado foi gravado em formato longo e mostra a coluna original utilizada. Para as planilhas descritas, a origem esperada é `percentual_de_alunos_alfabetizados_1`.


In [38]:
caminho_resultados = SILVER_DIR / "resultados_municipios.parquet"

if caminho_resultados.exists():
    validacao_resultados = pd.read_parquet(caminho_resultados)
    display(
        validacao_resultados[
            [
                coluna
                for coluna in (
                    "id_municipio",
                    "nome_municipio",
                    "sigla_uf",
                    "rede",
                    "ano",
                    "indicador_pct",
                    "_coluna_origem",
                    "_arquivo_fonte",
                    "_processed_at",
                )
                if coluna in validacao_resultados.columns
            ]
        ].head(20)
    )
    print("Colunas de origem:", sorted(validacao_resultados["_coluna_origem"].dropna().unique()))
else:
    print("Arquivo de resultados municipais ainda não foi gerado.")


,id_municipio,nome_municipio,sigla_uf,rede,ano,indicador_pct,_coluna_origem,_arquivo_fonte,_processed_at
0,3115607,Cedro do Abaeté,MG,MUNICIPAL,2023,76.47,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
1,3137908,Lamim,MG,MUNICIPAL,2023,100.0,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
2,3167103,Serro,MG,MUNICIPAL,2023,81.82,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
3,3547650,Santa Salete,SP,MUNICIPAL,2023,100.0,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
4,4300406,Alegrete,RS,MUNICIPAL,2023,53.93,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
5,4311155,Jóia,RS,MUNICIPAL,2023,60.87,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
6,4317251,Santa Tereza,RS,MUNICIPAL,2023,90.91,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
7,4320602,Severiano de Almeida,RS,MUNICIPAL,2023,73.33,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
8,4321477,Tiradentes do Sul,RS,MUNICIPAL,2023,90.67,percentual_de_alunos_alfabetizados_1,resultados_municipios_2023.xlsx,2026-11-03T02:38:55.338688+00:00
9,2412302,São José do Campestre,RN,MUNICIPAL,2023,18.1,percentual_de_alunos_alfabetizados_2023_1,resultados_municipios_2024.xlsx,2026-11-03T02:38:55.338688+00:00


Colunas de origem: ['percentual_de_alunos_alfabetizados_1', 'percentual_de_alunos_alfabetizados_2023_1', 'percentual_de_alunos_alfabetizados_2024_1', 'percentual_de_alunos_alfabetizados_2025_1']
